In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [21]:
actual = pd.read_csv('../data/raw/actual.csv')
test_df = pd.read_csv('../data/raw/data_set_ALL_AML_independent.csv')
train_df = pd.read_csv('../data/raw/data_set_ALL_AML_train.csv')

In [22]:
df1 = [col for col in train_df.columns if "call" not in col]
train_df = train_df[df1]
train_df.T.head()
train_df = train_df.T
train_df2 = train_df.drop(['Gene Description','Gene Accession Number'],axis=0)
train_df2.index = pd.to_numeric(train_df2.index)
train_df2.sort_index(inplace=True)
train_df2.head()

,0,1,2,3,4,5,6,7,8,9,...,7119,7120,7121,7122,7123,7124,7125,7126,7127,7128
1,-214,-153,-58,88,-295,-558,199,-176,252,206,...,185,511,-125,389,-37,793,329,36,191,-37
2,-139,-73,-1,283,-264,-400,-330,-168,101,74,...,169,837,-36,442,-17,782,295,11,76,-14
3,-76,-49,-307,309,-376,-650,33,-367,206,-215,...,315,1199,33,168,52,1138,777,41,228,-41
4,-135,-114,265,12,-419,-585,158,-253,49,31,...,240,835,218,174,-110,627,170,-50,126,-91
5,-106,-125,-76,168,-230,-284,4,-122,70,252,...,156,649,57,504,-26,250,314,14,56,-25


In [23]:
train_df2['cancer_type'] = list(pd.read_csv('../data/raw/actual.csv')[:38]['cancer'])
dic = {'ALL':0,'AML':1}
train_df2.replace(dic,inplace=True)

,0,1,2,3,4,5,6,7,8,9,...,7120,7121,7122,7123,7124,7125,7126,7127,7128,cancer_type
1,-214,-153,-58,88,-295,-558,199,-176,252,206,...,511,-125,389,-37,793,329,36,191,-37,0
2,-139,-73,-1,283,-264,-400,-330,-168,101,74,...,837,-36,442,-17,782,295,11,76,-14,0
3,-76,-49,-307,309,-376,-650,33,-367,206,-215,...,1199,33,168,52,1138,777,41,228,-41,0
4,-135,-114,265,12,-419,-585,158,-253,49,31,...,835,218,174,-110,627,170,-50,126,-91,0
5,-106,-125,-76,168,-230,-284,4,-122,70,252,...,649,57,504,-26,250,314,14,56,-25,0
6,-138,-85,215,71,-272,-558,67,-186,87,193,...,1221,-76,172,-74,645,341,26,193,-53,0
7,-72,-144,238,55,-399,-551,131,-179,126,-20,...,819,-178,151,-18,1140,482,10,369,-42,0
8,-413,-260,7,-2,-541,-790,-275,-463,70,-169,...,629,-86,302,23,1799,446,59,781,20,0
9,5,-127,106,268,-210,-535,0,-174,24,506,...,980,6,177,-12,758,385,115,244,-39,0
10,-88,-105,42,219,-178,-246,328,-148,177,183,...,986,26,101,21,570,359,9,171,7,0


Sparse Logistic Regression

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
X_std = StandardScaler().fit_transform(train_df2.drop('cancer_type',axis=1))
y = train_df2['cancer_type']

In [32]:
y_fit = y.to_numpy().ravel().astype(int)
logreg = LogisticRegression(
    l1_ratio=1,           # pure L1
    solver='saga', C=0.08,
    max_iter=10000, tol=1e-4, random_state=0
).fit(X_std, y_fit)

print('n_iter:', logreg.n_iter_)
print('nonzero coefs:', np.sum(logreg.coef_ != 0))

print(logreg.n_iter_)   # if this equals max_iter, it did NOT converge

logreg.predict(X_std)

score = logreg.score(X_std, y_fit)
print('score:',score)
coef = logreg.coef_.ravel()
nonzero_idx = np.where(coef != 0)[0]
print('nonzero coefs:', len(nonzero_idx))

n_iter: [2546]
nonzero coefs: 3
[2546]
score: 0.7105263157894737
nonzero coefs: 3
